In [2]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
import lightgbm as lgb
from src.evaluation import load_train, cross_validate, calibration_by_exposure, poisson_deviance, PROCESSED
from src.features import LADDER
from src.gbm import fit_predict, PARAMS

train = load_train()
CAT_RAW = ["VehBrand", "VehGas", "Region"]
BASE_F = ["VehPower", "VehAge", "DrivAge", "BonusMalus", "Density", "VehBrand", "VehGas", "Region"]

def make_X_exp(d, levels):
    X = d[BASE_F + ["Exposure"]].copy()
    X["bm_off_ladder"] = (~d["BonusMalus"].isin(LADDER)).astype(int)
    for c in CAT_RAW:
        X[c] = pd.Categorical(X[c], categories=levels[c])
    return X

def fit_predict_exp(tr, va):
    levels = {c: sorted(tr[c].unique()) for c in CAT_RAW}          # names from the fitting part only
    m = lgb.LGBMRegressor(**PARAMS)
    m.fit(make_X_exp(tr, levels), tr["ClaimNb"] / tr["Exposure"], sample_weight=tr["Exposure"])
    return m.predict(make_X_exp(va, levels)) * va["Exposure"].to_numpy()

oof_B, s_B = cross_validate(train, fit_predict_exp)
oof_A = pd.read_csv(PROCESSED / "oof_gbm.csv")["pred_claims"].to_numpy()
s_A = np.array([poisson_deviance(train.loc[train["Fold"] == f, "ClaimNb"], oof_A[(train["Fold"] == f).to_numpy()])
                for f in range(5)])

print("A (no exposure):", s_A.round(4), "| mean", round(s_A.mean(), 4))
print("B (+ exposure) :", s_B.round(4), "| mean", round(s_B.mean(), 4))
print("gain A - B per fold:", (s_A - s_B).round(4), "| folds B better:", int((s_B < s_A).sum()))
print("\noverall actual/predicted B:", round(train["ClaimNb"].sum() / oof_B.sum(), 4), "\n")
print(calibration_by_exposure(train, oof_B), "\n")

y = train["ClaimNb"]
age0 = train["VehAge"] == 0
rows = []
for thr in [0.75, 0.9, 0.99]:
    for name, grp in [("age 0", age0), ("age 1+", ~age0)]:
        s = ((train["Exposure"] >= thr) & grp).to_numpy()
        rows.append({"exposure >=": thr, "group": name, "claims": int(y[s].sum()),
                     "A actual/pred": round(y[s].sum() / oof_A[s].sum(), 3),
                     "B actual/pred": round(y[s].sum() / oof_B[s].sum(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

A (no exposure): [0.302  0.3038 0.3026 0.3095 0.3011] | mean 0.3038
B (+ exposure) : [0.289  0.2909 0.2903 0.2958 0.2872] | mean 0.2906
gain A - B per fold: [0.013  0.0129 0.0123 0.0137 0.0139] | folds B better: 5

overall actual/predicted B: 1.0016 

             policies  actual  predicted  actual/predicted
Exposure                                                  
(0.0, 0.1]     102054    2785     2745.9             1.014
(0.1, 0.25]     76054    2890     2908.5             0.994
(0.25, 0.5]    105108    5212     5220.3             0.998
(0.5, 0.75]     73967    4822     4772.4             1.010
(0.75, 1.0]    184249   13123    13139.1             0.999 

 exposure >=  group  claims  A actual/pred  B actual/pred
        0.75  age 0     405          0.427          0.937
        0.75 age 1+   13088          0.851          1.002
        0.90  age 0     110          0.384          0.898
        0.90 age 1+   10766          0.834          1.000
        0.99  age 0      11          0.307 

In [3]:
levels = {c: sorted(train[c].unique()) for c in CAT_RAW}
m = lgb.LGBMRegressor(**PARAMS)
m.fit(make_X_exp(train, levels), train["ClaimNb"] / train["Exposure"], sample_weight=train["Exposure"])

old = train[train["VehAge"] > 0].sample(20000, random_state=0)
for e in [0.1, 0.25, 0.5, 0.75, 1.0]:
    d = old.assign(Exposure=e)
    print(f"Exposure={e}: mean predicted rate (age 1+) = {m.predict(make_X_exp(d, levels)).mean():.4f}")

new = train[train["VehAge"] == 0].assign(Exposure=1.0)
print("age 0 at Exposure=1:", round(m.predict(make_X_exp(new, levels)).mean(), 4))

Exposure=0.1: mean predicted rate (age 1+) = 0.3043
Exposure=0.25: mean predicted rate (age 1+) = 0.1484
Exposure=0.5: mean predicted rate (age 1+) = 0.0987
Exposure=0.75: mean predicted rate (age 1+) = 0.0936
Exposure=1.0: mean predicted rate (age 1+) = 0.0783
age 0 at Exposure=1: 0.1207
